# 05. Variant 2: Pin-SVM

## Ý tưởng cần code: bài toán đối ngẫu của Pin-SVM giống hệt SVM, chỉ khác hộp ràng buộc: 0 ≤ αᵢ ≤ C thành −τC ≤ αᵢ ≤ C (Chương 2.2.3)

$$
\min_{\lambda}\quad
\frac{1}{2}\lambda^\top P\lambda + q^\top \lambda
$$

$$
\text{s.t.}\quad
G\lambda \leq h,\qquad A\lambda = b
$$


## import tools.py và models.py

In [1]:
import sys, json
sys.path.insert(0, "..")
import numpy as np, pandas as pd
from src.tools import load_features, tune_walk_forward, evaluate_on_test, summarize, train_test, scores
from src.models import PinSVM

data = load_features()

## Lưới tham số và chọn tham số

In [4]:
GRID = [
    {"kernel": ["linear"], "C": [0.01, 0.1, 1, 10], "tau": [0.1, 0.5, 1.0]},
    {"kernel": ["rbf"], "C": [0.1, 1, 10], "tau": [0.1, 0.5, 1.0], "gamma": [0.01, 0.05, 0.2]},
]
best, rows = tune_walk_forward(PinSVM, GRID, data.loc[:"2022"])      # chạy khoảng 5-6 phút
json.dump({"best": best, "rows": rows}, open("../reports/tables/tuning_Pin-SVM.json", "w"), indent=1)
print("Tham số tốt nhất:", best)

pd.DataFrame([{**r["params"], "BalAcc_val": r["BalAcc"], "std": r["std"]} for r in rows]) \
  .sort_values("BalAcc_val", ascending=False).head(8)

Tham số tốt nhất: {'kernel': 'rbf', 'C': 1, 'tau': 1.0, 'gamma': 0.01}


,kernel,C,tau,BalAcc_val,std,gamma
27,rbf,1.00,1.0,0.523431,0.032472,0.01
24,rbf,1.00,0.5,0.522807,0.027328,0.01
1,linear,0.01,0.5,0.516134,0.053041,NaN
13,rbf,0.10,0.1,0.515962,0.019255,0.05
21,rbf,1.00,0.1,0.515389,0.028618,0.01
19,rbf,0.10,1.0,0.515179,0.019595,0.05
0,linear,0.01,0.1,0.513612,0.045948,NaN
12,rbf,0.10,0.1,0.512556,0.033315,0.01


## Đánh giá trên test

In [5]:
res = evaluate_on_test("Pin-SVM", PinSVM, best, data)
res.to_csv("../reports/tables/test_Pin-SVM.csv", index=False)
summarize(res, ["Accuracy", "BalAcc", "F1", "AUC", "MCC", "UpRate", "time_s", "nonzero"])

,Accuracy,BalAcc,F1,AUC,MCC,UpRate,time_s,nonzero
model,,,,,,,,
Pin-SVM,0.5234,0.5080,0.5955,0.5061,0.0161,0.5997,1.0950,1.0000


## Anh hưởng của τ, và phép thử τ = 0

In [6]:
X_train, y_train, X_test, y_test, *_ = train_test(data)
rows_tau = []
for tau in [0.0, 0.1, 0.3, 0.5, 1.0]:
    m = PinSVM(**{**best, "tau": tau}).fit(X_train, y_train)
    rows_tau.append({"tau": tau,
                     "nonzero": m.nonzero_ratio_,
                     "alpha<0 (điểm đúng bị phạt)": np.mean(m.lam_ < -1e-6 * best["C"]),
                     "BalAcc_test": scores(y_test, m.predict(X_test))["BalAcc"]})
tau_df = pd.DataFrame(rows_tau)
tau_df.to_csv("../reports/tables/pin_tau.csv", index=False)
tau_df

,tau,nonzero,alpha<0 (điểm đúng bị phạt),BalAcc_test
0,0.0,0.986713,0.000000,0.513624
1,0.1,1.000000,0.034662,0.510880
2,0.3,1.000000,0.030618,0.508565
3,0.5,1.000000,0.027730,0.508565
4,1.0,1.000000,0.021375,0.508003
